In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 07 One-Click LexAI Runner (Final Stable)

Run cells top-to-bottom once.

Design goals:
- No hard failures across cells (graceful skip if prerequisite is missing)
- Databricks workspace notebook path handling via adapter fallback
- Correct browser links (driver-proxy), not localhost links
- Optional non-blocking Streamlit startup  Repos


In [0]:
# CELL 0.5: Optional Python restart (disabled by default)
# Set to True only when dependency stack is broken.
FORCE_PYTHON_RESTART = False

if FORCE_PYTHON_RESTART:
    print("[CELL 0.5] Restarting Python runtime...")
    dbutils.library.restartPython()
else:
    print("[CELL 0.5] Skipping restart (one-click mode).")


[CELL 0.5] Skipping restart (one-click mode).


In [0]:
# CELL 1: Runtime Flags
AUTO_INSTALL_MISSING = True
RUN_SMOKE_TEST = True
START_FASTAPI = True
START_STREAMLIT = False

# Only use this if import stack is broken and you accept manual restart+rerun.
FORCE_REPAIR_IMPORT_STACK = False

# Prefer non-default ports to avoid collisions on shared serverless hosts.
FASTAPI_PORT = 8765
STREAMLIT_PORT = 8501

# Optional override if org id auto-detection fails (from URL ?o=<org_id>)
ORG_ID_OVERRIDE = "7474658388963127"

# Optional override for repo root. Keep empty for auto-detection.
REPO_DIR_OVERRIDE = "/Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform"

# Keep services alive after Run All. Set True only when you want to stop at the end.
STOP_SERVERS_AT_END = False

SMOKE_TEST_QUERIES = [
    "Penalty for not wearing helmet in short within 120 words",
    "What does section 129 say in detail within 180 words",
]

# Shared runtime flags
REPO_OK = False
DEPS_OK = False
ENGINE_READY = False
API_READY = False
STREAMLIT_READY = False

print("[CELL 1] Flags loaded")
print({
    "AUTO_INSTALL_MISSING": AUTO_INSTALL_MISSING,
    "RUN_SMOKE_TEST": RUN_SMOKE_TEST,
    "START_FASTAPI": START_FASTAPI,
    "START_STREAMLIT": START_STREAMLIT,
    "FORCE_REPAIR_IMPORT_STACK": FORCE_REPAIR_IMPORT_STACK,
    "FASTAPI_PORT": FASTAPI_PORT,
    "STREAMLIT_PORT": STREAMLIT_PORT,
    "ORG_ID_OVERRIDE": ORG_ID_OVERRIDE,
    "STOP_SERVERS_AT_END": STOP_SERVERS_AT_END,
    "REPO_DIR_OVERRIDE": REPO_DIR_OVERRIDE,
})


[CELL 1] Flags loaded
{'AUTO_INSTALL_MISSING': True, 'RUN_SMOKE_TEST': True, 'START_FASTAPI': True, 'START_STREAMLIT': False, 'FORCE_REPAIR_IMPORT_STACK': False, 'FASTAPI_PORT': 8765, 'STREAMLIT_PORT': 8501, 'ORG_ID_OVERRIDE': '7474658388963127', 'STOP_SERVERS_AT_END': False, 'REPO_DIR_OVERRIDE': '/Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform'}


In [0]:
# CELL 2: Resolve repo path safely
import os
import sys
from pathlib import Path
from datetime import datetime


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def _repo_has_required_files(repo_dir: Path) -> bool:
    return (
        (repo_dir / "apps" / "fastapi_app.py").exists()
        and (repo_dir / "apps" / "lexai06_notebook_adapter.py").exists()
    )


def _safe_walk_for_repo(root: Path):
    skip_dirs = {"__pycache__", ".git", ".ipynb_checkpoints"}

    def _onerror(_err):
        return None

    for dirpath, dirnames, _ in os.walk(root, topdown=True, onerror=_onerror):
        dirnames[:] = [d for d in dirnames if d not in skip_dirs]
        p = Path(dirpath)
        try:
            if _repo_has_required_files(p):
                return p
        except Exception:
            continue
    return None


def _context_repo_guess():
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        nb_path = ctx.notebookPath().get()  # /Users/<email>/<repo>/notebooks/...
        if not nb_path:
            return None
        pp = Path(nb_path)
        ws_repo = Path("/Workspace") / Path(*pp.parent.parent.parts[1:])
        if ws_repo.exists() and _repo_has_required_files(ws_repo):
            return ws_repo
    except Exception:
        pass
    return None


def resolve_repo_dir() -> Path:
    if REPO_DIR_OVERRIDE and str(REPO_DIR_OVERRIDE).strip():
        p = Path(REPO_DIR_OVERRIDE.strip())
        if p.exists() and _repo_has_required_files(p):
            return p

    cwd = Path(os.getcwd()).resolve()
    for cand in [cwd] + list(cwd.parents):
        try:
            if _repo_has_required_files(cand):
                return cand
        except Exception:
            continue

    g = _context_repo_guess()
    if g is not None:
        return g

    for root in [Path("/Workspace/Repos"), Path("/Workspace/Users"), Path("/Workspace")]:
        if not root.exists():
            continue
        hit = _safe_walk_for_repo(root)
        if hit is not None:
            return hit

    return Path(os.getcwd()).resolve()


REPO_DIR = resolve_repo_dir()
if _repo_has_required_files(REPO_DIR):
    REPO_OK = True
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    log(f"Repo root: {REPO_DIR}")
    print("[CELL 2] REPO_OK=True")
else:
    REPO_OK = False
    print("[CELL 2] REPO_OK=False; unresolved repo path:", REPO_DIR)
    print("[CELL 2] Set REPO_DIR_OVERRIDE to your repo path and rerun from Cell 1.")


[13:19:16] Repo root: /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform
[CELL 2] REPO_OK=True


In [0]:
# CELL 3: Dependency preflight - Databricks serverless-safe
import os
import sys
import importlib
import subprocess

DEPS_OK = False

if not REPO_OK:
    print("[CELL 3] Skipped: REPO_OK=False")
else:
    print("[CELL 3] Checking dependencies...")

    # Reduce optional-backend import issues in serverless sessions.
    os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
    os.environ.setdefault("TRANSFORMERS_NO_FLAX", "1")
    os.environ.setdefault("USE_TF", "0")
    os.environ.setdefault("USE_FLAX", "0")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

    required = [
        ("fastapi", "fastapi"),
        ("uvicorn", "uvicorn"),
        ("requests", "requests"),
        ("databricks.sdk", "databricks-sdk"),
        ("mlflow", "mlflow"),
        ("numpy", "numpy"),
        ("sentence_transformers", "sentence-transformers"),
    ]

    missing_pkgs = []
    for mod, pkg in required:
        try:
            importlib.import_module(mod)
        except Exception:
            missing_pkgs.append(pkg)

    # Keep transformer stack explicit for stability when sentence-transformers loads.
    extra_runtime_pkgs = ["transformers>=4.30.0", "accelerate>=0.20.0", "typing_extensions>=4.6.0"]

    if missing_pkgs:
        print("[CELL 3] Missing packages:", missing_pkgs)
        if AUTO_INSTALL_MISSING:
            cmd = [sys.executable, "-m", "pip", "install", "-q"] + missing_pkgs + extra_runtime_pkgs
            try:
                print("[CELL 3] Installing missing packages...")
                subprocess.check_call(cmd)
                print("[CELL 3] Install completed")
            except Exception as e:
                print("[CELL 3] Install failed:", e)
        else:
            print("[CELL 3] AUTO_INSTALL_MISSING=False")

    # Clear possibly half-initialized imports from previous failed attempts.
    for k in list(sys.modules.keys()):
        if k.startswith(("accelerate", "transformers", "sentence_transformers", "jax", "flax", "tensorflow", "keras")):
            del sys.modules[k]

    try:
        import fastapi  # noqa: F401
        import uvicorn  # noqa: F401
        import requests  # noqa: F401
        import databricks.sdk  # noqa: F401
        import mlflow  # noqa: F401
        import numpy  # noqa: F401
        import sentence_transformers  # noqa: F401
        DEPS_OK = True
        print("[CELL 3] DEPS_OK=True")
    except Exception as e:
        print("[CELL 3] Final dependency validation failed:", e)
        DEPS_OK = False

        if FORCE_REPAIR_IMPORT_STACK:
            print("[CELL 3] FORCE_REPAIR_IMPORT_STACK=True -> running repair install")
            repair_cmd = [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--upgrade",
                "typing_extensions>=4.6.0",
                "accelerate>=0.20.0",
                "transformers>=4.30.0",
                "sentence-transformers>=2.6.0",
                "databricks-sdk",
            ]
            try:
                subprocess.check_call(repair_cmd)
                print("[CELL 3] Repair install done. If imports still fail, restart Python and rerun from Cell 1.")
            except Exception as e2:
                print("[CELL 3] Repair install failed:", e2)


[CELL 3] Checking dependencies...
[CELL 3] DEPS_OK=True


In [0]:
# CELL 4: Spark context and browser-link base
from pyspark.sql import SparkSession
import json
import re

cluster_id = "unknown"
org_id = "unknown"
workspace_url = "unknown"
compute_mode = "unknown"
DRIVER_PROXY_BASE = ""
DRIVER_PROXY_SUPPORTED = False

try:
    spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
    print("[CELL 4] Spark session ready:", bool(spark))
except Exception as e:
    spark = None
    print("[CELL 4] Spark unavailable:", e)


def _safe_conf(key: str, default: str = ""):
    if spark is None:
        return default
    try:
        v = spark.conf.get(key)
        if v and str(v).strip():
            return str(v).strip()
    except Exception:
        pass
    return default


def _opt_to_str(opt):
    try:
        if hasattr(opt, "isDefined") and opt.isDefined():
            return str(opt.get())
    except Exception:
        pass
    try:
        return str(opt.get())
    except Exception:
        pass
    return ""


def _ctx_tag(ctx, key: str):
    try:
        tags = ctx.tags()
        return _opt_to_str(tags.get(key))
    except Exception:
        pass
    try:
        tags = ctx.tags()
        return str(tags.apply(key))
    except Exception:
        pass
    return ""


def _ctx_json(ctx):
    try:
        raw = _opt_to_str(ctx.toJson()) or str(ctx.toJson())
        if raw and raw.strip().startswith("{"):
            return json.loads(raw)
    except Exception:
        pass
    return {}


def _extract_org_from_text(text: str) -> str:
    if not text:
        return ""
    m = re.search(r"[?&]o=(\d+)", text)
    if m:
        return m.group(1)
    m = re.search(r"\"orgId\"\s*:\s*\"?(\d+)\"?", text)
    if m:
        return m.group(1)
    return ""


ctx = None
try:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
except Exception:
    pass

ctxj = _ctx_json(ctx) if ctx else {}
ctx_tags = ctxj.get("tags", {}) if isinstance(ctxj, dict) else {}

cluster_id = (
    _safe_conf("spark.databricks.clusterUsageTags.clusterId", "")
    or _ctx_tag(ctx, "clusterId")
    or str(ctx_tags.get("clusterId", ""))
    or "unknown"
)

org_id = (
    (str(ORG_ID_OVERRIDE).strip() if str(ORG_ID_OVERRIDE).strip() else "")
    or _safe_conf("spark.databricks.clusterUsageTags.orgId", "")
    or _ctx_tag(ctx, "orgId")
    or str(ctx_tags.get("orgId", ""))
    or _extract_org_from_text(json.dumps(ctxj))
    or "unknown"
)

compute_mode = (
    _safe_conf("spark.databricks.clusterUsageTags.clusterSource", "")
    or _ctx_tag(ctx, "clusterSource")
    or str(ctx_tags.get("clusterSource", ""))
    or "unknown"
)

ws_from_conf = _safe_conf("spark.databricks.workspaceUrl", "")
ws_from_ctx = _opt_to_str(ctx.browserHostName()) if ctx else ""
ws_api = _opt_to_str(ctx.apiUrl()) if ctx else ""
workspace_url = ws_from_conf or ws_from_ctx
if (not workspace_url) and ws_api:
    workspace_url = ws_api.replace("https://", "").split("/")[0]
if not workspace_url:
    workspace_url = "unknown"

if workspace_url != "unknown" and org_id != "unknown" and cluster_id != "unknown":
    DRIVER_PROXY_BASE = f"https://{workspace_url}/driver-proxy/o/{org_id}/{cluster_id}"
    DRIVER_PROXY_SUPPORTED = True

print("[CELL 4] cluster_id:", cluster_id)
print("[CELL 4] org_id:", org_id)
print("[CELL 4] workspace_url:", workspace_url)
print("[CELL 4] compute_mode:", compute_mode)
print("[CELL 4] DRIVER_PROXY_SUPPORTED:", DRIVER_PROXY_SUPPORTED)
print("[CELL 4] DRIVER_PROXY_BASE:", DRIVER_PROXY_BASE or "unavailable")

if not DRIVER_PROXY_SUPPORTED:
    print("[CELL 4] If org_id is unknown, set ORG_ID_OVERRIDE in Cell 1.")
    print("[CELL 4] You can get org id from Databricks URL query param: ?o=<org_id>")


[CELL 4] Spark session ready: True


2026-03-02 13:19:18,664 31226 ERROR _handle_rpc_error GRPC Error received
Traceback (most recent call last):
  File "/databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/client/core.py", line 1724, in config
    resp = self._stub.Config(req, metadata=self.metadata())
  File "/databricks/python/lib/python3.10/site-packages/grpc/_interceptor.py", line 277, in __call__
    response, ignored_call = self._with_call(
  File "/databricks/python/lib/python3.10/site-packages/grpc/_interceptor.py", line 332, in _with_call
    return call.result(), call
  File "/databricks/python/lib/python3.10/site-packages/grpc/_channel.py", line 439, in result
    raise self
  File "/databricks/python/lib/python3.10/site-packages/grpc/_interceptor.py", line 315, in continuation
    response, call = self._thunk(new_method).with_call(
  File "/databricks/python/lib/python3.10/site-packages/grpc/_channel.py", line 1193, in with_call
    return _end_unary_response_blocking(state, call, True, None)
 

[CELL 4] cluster_id: 0302-051116-1ns9zuy8-v2n
[CELL 4] org_id: 7474658388963127
[CELL 4] workspace_url: dbc-afb2e98d-d930.cloud.databricks.com
[CELL 4] compute_mode: unknown
[CELL 4] DRIVER_PROXY_SUPPORTED: True
[CELL 4] DRIVER_PROXY_BASE: https://dbc-afb2e98d-d930.cloud.databricks.com/driver-proxy/o/7474658388963127/0302-051116-1ns9zuy8-v2n


In [0]:
# CELL 4.5: Optional repair marker
if FORCE_REPAIR_IMPORT_STACK and not DEPS_OK:
    print("[CELL 4.5] Repair mode active and DEPS_OK=False.")
    print("[CELL 4.5] If you ran repair installs, run dbutils.library.restartPython() and rerun from Cell 1.")
else:
    print("[CELL 4.5] No repair action needed.")


[CELL 4.5] No repair action needed.


In [0]:
# CELL 5: Initialize notebook-06 engine through adapter
import os
from pathlib import Path
import importlib

ENGINE_READY = False
status = {}
last_err = None

print("[CELL 5] REPO_OK:", REPO_OK)
print("[CELL 5] DEPS_OK:", DEPS_OK)
print("[CELL 5] REPO_DIR:", REPO_DIR)

if not REPO_OK:
    print("[CELL 5] Cannot proceed: REPO_OK=False")
elif not DEPS_OK:
    print("[CELL 5] Cannot proceed: DEPS_OK=False")
else:
    import apps.lexai06_notebook_adapter as _adapter
    importlib.reload(_adapter)
    NotebookEngine = _adapter.NotebookEngine

    repo_dir = Path(REPO_DIR)

    # Prefer live notebook 06 first (latest logic), then snapshot fallback for serverless stability.
    snapshot_json = repo_dir / "apps" / "notebook_06_snapshot.json"
    snapshot_ipynb = repo_dir / "apps" / "notebook_06_snapshot.ipynb"
    snapshot_obj = repo_dir / "apps" / "notebook_06_snapshot"
    notebook_ipynb = repo_dir / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine.ipynb"
    notebook_obj = repo_dir / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine"

    candidates = [
        notebook_obj,
        notebook_ipynb,
        snapshot_obj,
        snapshot_ipynb,
        snapshot_json,
    ]

    # Add workspace-style paths from notebook context (/Repos/...) for export-based fallback.
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        ws_nb_path = ctx.notebookPath().get()  # /Repos/<email>/<repo>/notebooks/07_...
        ws_nb_dir = Path(ws_nb_path).parent
        candidates.extend([
            ws_nb_dir / "06_High-precision_QA_Legal_Reasoning_Engine",
            ws_nb_dir.parent / "apps" / "notebook_06_snapshot",
            ws_nb_dir.parent / "apps" / "notebook_06_snapshot.json",
        ])
    except Exception:
        pass

    # De-duplicate preserving order.
    unique_candidates = []
    seen = set()
    for c in candidates:
        cs = str(c)
        if cs in seen:
            continue
        seen.add(cs)
        unique_candidates.append(c)

    print("[CELL 5] Notebook candidates:")
    for c in unique_candidates:
        try:
            print(" -", c, "exists=", Path(c).exists())
        except Exception:
            print(" -", c, "exists=ERROR")

    for cand in unique_candidates:
        try:
            os.environ["LEXAI06_NOTEBOOK_PATH"] = str(cand)
            engine = NotebookEngine(notebook_path=Path(str(cand)))
            status = engine.initialize()
            ENGINE_READY = bool(status.get("ready", False))
            if ENGINE_READY:
                print(f"[CELL 5] Initialized using candidate: {cand}")
                break
            print(f"[CELL 5] Candidate returned ready=False: {cand}")
        except Exception as e:
            last_err = e
            print(f"[CELL 5] Candidate failed: {cand} -> {e}")

    if ENGINE_READY:
        print("[CELL 5] Engine initialized")
        for k, v in status.items():
            print(f"  - {k}: {v}")
    else:
        print("[CELL 5] Engine not ready. Last error:", last_err)

print("[CELL 5] ENGINE_READY =", ENGINE_READY)


[CELL 5] REPO_OK: True
[CELL 5] DEPS_OK: True
[CELL 5] REPO_DIR: /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform
[CELL 5] Notebook candidates:
 - /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine exists= True
 - /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine.ipynb exists= False
 - /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot exists= True
 - /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot.ipynb exists= False
 - /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot.json exists= True
 - /Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine exists= False
 - /Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snaps

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BertModel LOAD REPORT from: sentence-transformers

[13:19:26] Embedding model ready: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/a

[13:19:27] Using endpoint backend: databricks-meta-llama-3-3-70b-instruct
[13:19:35] Loaded lexical artifacts from Delta for signature=312b53f785d939bab1c3.
--- Runtime Status ---
Data signature: 312b53f785d939bab1c3
Embedding source: /Volumes/workspace/legal_data/vector_db_test/legal_embeddings_delta
Records loaded: 5194
Embedding dim: 384
Avg doc len: 183.25
Vocabulary size: 13196
Lexical source: delta_artifact
Embedder: sentence-transformers/all-MiniLM-L6-v2
Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
LLM backend: endpoint (databricks-meta-llama-3-3-70b-instruct)
[CELL 5] Initialized using candidate: /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine
[CELL 5] Engine initialized
  - ready: True
  - error: 
  - notebook_path: /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine
  - resolved_notebook_path: /Workspace/Repos/kumarangad54706@gmail.

In [0]:
# CELL 6: Smoke test
if RUN_SMOKE_TEST and ENGINE_READY:
    print("[CELL 6] Running smoke tests...")
    for idx, q in enumerate(SMOKE_TEST_QUERIES, start=1):
        print("=" * 90)
        print(f"[{idx}] Query: {q}")
        out = engine.answer_query(q)
        print("Mode:", out.get("mode"))
        print("Source:", out.get("source"))
        print("Confidence:", out.get("confidence"))
        print("Sections:", out.get("sections", []))
        print("Citations:", out.get("citations", [])[:5])
        print("Latency:", out.get("latency_ms", {}))
        print("Answer:")
        print(out.get("answer", ""))
    print("[CELL 6] Smoke tests done")
else:
    print("[CELL 6] Skipped (RUN_SMOKE_TEST or ENGINE_READY condition not met).")


[CELL 6] Running smoke tests...
[1] Query: Penalty for not wearing helmet in short within 120 words
Mode: rule_based
Source: traffic_rules
Confidence: top_score=0.617, avg_lex=0.732, coverage=0.444, normative_hits=5, judgment_hits=0, candidates=2512, shortlist=140, route=statute_strict
Sections: ['177', '178', '179', '180', '181', '182', '183', '184', '186', '189', '190', '191', '129']
Citations: ['Motor Vehicles Act 1988 - Section Chapter V', 'Motor Vehicles Act 1988 - Section Chapter VII', 'Motor Vehicle Ammendment Act 2019 - Section Chapter XI', 'Motor Vehicles Act 1988 - Section Chapter VI', 'Central Motor Vehicle Rules 1989 - Section Chapter VI']
Latency: {'candidate_fetch_ms': 3.8, 'routing_ms': 73.38, 'lexical_ms': 135.14, 'embed_ms': 69.52, 'dense_ms': 7.45, 'rrf_ms': 0.27, 'rerank_ms': 1360.28, 'retrieve_total_ms': 1653.37, 'generation_ms': 0.0, 'total_ms': 1655.56}
Answer:
Law:
Section 129 of the Motor Vehicles Act requires riders to wear protective headgear while riding two-

In [0]:
# CELL 7: Start FastAPI and print exact browser links
import os
import sys
import subprocess
import time
from pathlib import Path
import requests

API_READY = False
API_BROWSER_HEALTH_URL = ""
API_BROWSER_DOCS_URL = ""

print("[CELL 7] START_FASTAPI =", START_FASTAPI)
print("[CELL 7] ENGINE_READY =", ENGINE_READY)

FASTAPI_PROC = globals().get("FASTAPI_PROC")

# Ensure localhost calls never go through corporate/workspace HTTP proxies.
os.environ["NO_PROXY"] = "127.0.0.1,localhost"
os.environ["no_proxy"] = "127.0.0.1,localhost"

LOCAL_HTTP = requests.Session()
LOCAL_HTTP.trust_env = False


def _health_json(port: int, timeout_sec: int = 3):
    url = f"http://127.0.0.1:{port}/health"
    try:
        r = LOCAL_HTTP.get(url, timeout=timeout_sec)
        if r.status_code == 200:
            try:
                return True, r.json()
            except Exception:
                return True, {}
    except Exception:
        pass
    return False, {}


def _is_lexai_health(payload):
    return isinstance(payload, dict) and ("ok" in payload) and ("status" in payload)


def _wait_lexai(port: int, timeout_sec: int = 75):
    start = time.time()
    while (time.time() - start) < timeout_sec:
        ok, body = _health_json(port, timeout_sec=3)
        if ok and _is_lexai_health(body):
            return True, body
        time.sleep(1)
    return False, {}


def _candidate_ports(base_port: int):
    seeds = [
        base_port,
        base_port + 1,
        base_port + 2,
        8765,
        8766,
        8787,
        8800,
        8899,
        9000,
        9010,
    ]
    out = []
    seen = set()
    for p in seeds:
        p = int(p)
        if p not in seen:
            seen.add(p)
            out.append(p)
    for p in range(10080, 10120):
        if p not in seen:
            seen.add(p)
            out.append(p)
    return out


def _tail_file(path: Path, lines: int = 30):
    try:
        if not path.exists():
            return ""
        txt = path.read_text(encoding="utf-8", errors="ignore")
        arr = txt.splitlines()
        return "\n".join(arr[-lines:])
    except Exception:
        return ""


if START_FASTAPI and ENGINE_READY:
    # Reuse configured port if LexAI API is already up.
    ok_existing, body_existing = _health_json(int(FASTAPI_PORT), timeout_sec=3)
    if ok_existing and _is_lexai_health(body_existing):
        API_READY = True
        print(f"[CELL 7] Reusing existing LexAI API on port {FASTAPI_PORT}")

    # If tracked subprocess exists but current port is not healthy, drop stale handle.
    if (not API_READY) and (FASTAPI_PROC is not None):
        try:
            if FASTAPI_PROC.poll() is not None:
                FASTAPI_PROC = None
                globals()["FASTAPI_PROC"] = None
        except Exception:
            FASTAPI_PROC = None
            globals()["FASTAPI_PROC"] = None

    if not API_READY:
        started = False
        for p in _candidate_ports(int(FASTAPI_PORT)):
            ok_p, body_p = _health_json(p, timeout_sec=2)
            if ok_p and _is_lexai_health(body_p):
                FASTAPI_PORT = int(p)
                API_READY = True
                started = True
                print(f"[CELL 7] Found existing LexAI API on port {FASTAPI_PORT}")
                break

            log_path = Path("/tmp") / f"lexai_fastapi_{int(p)}.log"
            try:
                lf = open(log_path, "w", encoding="utf-8")
            except Exception:
                lf = None

            cmd = [
                sys.executable,
                "-m",
                "uvicorn",
                "apps.fastapi_app:app",
                "--host",
                "0.0.0.0",
                "--port",
                str(int(p)),
                "--workers",
                "1",
                "--log-level",
                "info",
            ]

            env = os.environ.copy()
            env.setdefault("PYTHONUNBUFFERED", "1")
            env["NO_PROXY"] = "127.0.0.1,localhost"
            env["no_proxy"] = "127.0.0.1,localhost"
            env["PYTHONPATH"] = f"{REPO_DIR}:{env.get('PYTHONPATH','')}"

            try:
                proc = subprocess.Popen(
                    cmd,
                    cwd=str(REPO_DIR),
                    env=env,
                    stdout=lf if lf is not None else subprocess.DEVNULL,
                    stderr=lf if lf is not None else subprocess.DEVNULL,
                )
            except Exception as e:
                if lf is not None:
                    lf.close()
                print(f"[CELL 7] Failed launching subprocess on port {p}: {e}")
                continue

            ok, body = _wait_lexai(int(p), timeout_sec=75)
            if ok:
                FASTAPI_PORT = int(p)
                API_READY = True
                started = True
                FASTAPI_PROC = proc
                globals()["FASTAPI_PROC"] = proc
                globals()["FASTAPI_LOG_PATH"] = str(log_path)
                print(f"[CELL 7] FastAPI started on port {FASTAPI_PORT}")
                if lf is not None:
                    lf.close()
                break

            rc = proc.poll()
            try:
                if proc.poll() is None:
                    proc.terminate()
                    time.sleep(1)
                    if proc.poll() is None:
                        proc.kill()
            except Exception:
                pass
            if lf is not None:
                lf.close()

            tail = _tail_file(log_path, lines=20)
            print(f"[CELL 7] Port {p} did not become healthy in time (returncode={rc})")
            if tail:
                print(f"[CELL 7] Log tail for port {p}:\n{tail}")

        if not started and not API_READY:
            print("[CELL 7] FastAPI could not be started on available candidate ports.")

    ok_final, body_final = _wait_lexai(int(FASTAPI_PORT), timeout_sec=6)
    API_READY = bool(ok_final)
    print("[CELL 7] Local health check:", "OK" if API_READY else "FAILED")
    if body_final:
        print(body_final)

    if API_READY:
        print("[CELL 7] Local-only URL (do not open in browser tab):", f"http://127.0.0.1:{FASTAPI_PORT}/health")
        if DRIVER_PROXY_SUPPORTED:
            API_BROWSER_HEALTH_URL = f"{DRIVER_PROXY_BASE}/{FASTAPI_PORT}/health"
            API_BROWSER_DOCS_URL = f"{DRIVER_PROXY_BASE}/{FASTAPI_PORT}/docs"
            print("[CELL 7] Browser URL (health):", API_BROWSER_HEALTH_URL)
            print("[CELL 7] Browser URL (docs):", API_BROWSER_DOCS_URL)
            try:
                displayHTML(f'<a href="{API_BROWSER_DOCS_URL}" target="_blank">Open FastAPI Docs</a>')
            except Exception:
                pass
        else:
            print("[CELL 7] Driver proxy not supported. Set ORG_ID_OVERRIDE in Cell 1 and rerun Cell 4 + Cell 7.")
    else:
        print("[CELL 7] API is not ready; browser URLs not printed to avoid 404 confusion.")

elif START_FASTAPI and not ENGINE_READY:
    print("[CELL 7] Skipped: ENGINE_READY=False. Fix Cell 5 first.")
else:
    print("[CELL 7] START_FASTAPI=False - skipping.")


[CELL 7] START_FASTAPI = True
[CELL 7] ENGINE_READY = True
[CELL 7] Port 8765 did not become healthy in time
[CELL 7] Port 8766 did not become healthy in time
[CELL 7] Port 8767 did not become healthy in time
[CELL 7] Port 8787 did not become healthy in time
[CELL 7] Port 8800 did not become healthy in time


In [0]:
# CELL 8: FastAPI smoke call
import requests

if API_READY:
    try:
        h = requests.get(f"http://127.0.0.1:{FASTAPI_PORT}/health", timeout=30)
        print("[CELL 8] local /health status:", h.status_code)
        print(h.json())

        payload = {
            "query": "What is the penalty for not wearing a helmet?",
            "style": "short",
            "word_limit": 120,
        }
        r = requests.post(f"http://127.0.0.1:{FASTAPI_PORT}/v1/legal/answer", json=payload, timeout=180)
        print("[CELL 8] local /v1/legal/answer status:", r.status_code)
        try:
            body = r.json()
            print("[CELL 8] answer preview:", body.get("answer", "")[:500])
        except Exception:
            print("[CELL 8] raw response:", r.text[:500])

        if API_BROWSER_DOCS_URL:
            print("[CELL 8] Browser docs URL:", API_BROWSER_DOCS_URL)
    except Exception as e:
        print("[CELL 8] API call failed:", e)
else:
    print("[CELL 8] Skipped: API_READY=False")


[CELL 8] Skipped: API_READY=False


In [0]:
# CELL 9: Optional Streamlit start (non-blocking) and browser URL
import subprocess

STREAMLIT_READY = False
STREAMLIT_BROWSER_URL = ""
STREAMLIT_PROC = globals().get("STREAMLIT_PROC")

if START_STREAMLIT and API_READY:
    if STREAMLIT_PROC is not None and STREAMLIT_PROC.poll() is None:
        STREAMLIT_READY = True
        print(f"[CELL 9] Streamlit already running on port {STREAMLIT_PORT}")
    else:
        selected_port = int(STREAMLIT_PORT)
        started = False
        for p in [selected_port, selected_port + 1, selected_port + 2]:
            try:
                cmd = [
                    sys.executable,
                    "-m",
                    "streamlit",
                    "run",
                    "apps/streamlit_app.py",
                    "--server.port", str(p),
                    "--server.address", "0.0.0.0",
                ]
                env = os.environ.copy()
                env["LEXAI_API_BASE_URL"] = f"http://127.0.0.1:{FASTAPI_PORT}"
                proc = subprocess.Popen(cmd, env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                STREAMLIT_PROC = proc
                STREAMLIT_PORT = int(p)
                STREAMLIT_READY = True
                globals()["STREAMLIT_PROC"] = STREAMLIT_PROC
                started = True
                print(f"[CELL 9] Streamlit started on port {STREAMLIT_PORT}")
                break
            except Exception:
                continue

        if not started:
            print("[CELL 9] Streamlit failed to start on tried ports.")

    if STREAMLIT_READY and DRIVER_PROXY_SUPPORTED:
        STREAMLIT_BROWSER_URL = f"{DRIVER_PROXY_BASE}/{STREAMLIT_PORT}/"
        print("[CELL 9] Browser Streamlit URL:", STREAMLIT_BROWSER_URL)
        try:
            displayHTML(f'<a href="{STREAMLIT_BROWSER_URL}" target="_blank">Open Streamlit UI</a>')
        except Exception:
            pass
    elif STREAMLIT_READY:
        print("[CELL 9] Streamlit running, but driver proxy unsupported for browser access in this context.")
else:
    print("[CELL 9] Skipped (START_STREAMLIT or API_READY condition not met).")


[CELL 9] Skipped (START_STREAMLIT or API_READY condition not met).


In [0]:
# CELL 10: Stop helper
if not STOP_SERVERS_AT_END:
    print("[CELL 10] STOP_SERVERS_AT_END=False -> keeping FastAPI/Streamlit running.")
    print("[CELL 10] Set STOP_SERVERS_AT_END=True in Cell 1 and rerun Cell 10 when you want to stop services.")
else:
    # Stop subprocess-based FastAPI if used.
    if "FASTAPI_PROC" in globals() and globals().get("FASTAPI_PROC") is not None:
        proc = globals()["FASTAPI_PROC"]
        try:
            if proc.poll() is None:
                proc.terminate()
                print("[CELL 10] FastAPI subprocess termination requested")
            else:
                print("[CELL 10] FastAPI subprocess already stopped")
        except Exception as e:
            print("[CELL 10] FastAPI subprocess stop failed:", e)
    # Backward compatibility: stop thread-based server if present.
    elif "FASTAPI_SERVER" in globals() and globals().get("FASTAPI_SERVER") is not None:
        globals()["FASTAPI_SERVER"].should_exit = True
        print("[CELL 10] FastAPI thread stop requested")
    else:
        print("[CELL 10] FastAPI was not running")

    if "STREAMLIT_PROC" in globals() and globals().get("STREAMLIT_PROC") is not None:
        proc = globals()["STREAMLIT_PROC"]
        try:
            if proc.poll() is None:
                proc.terminate()
                print("[CELL 10] Streamlit process termination requested")
            else:
                print("[CELL 10] Streamlit process already stopped")
        except Exception as e:
            print("[CELL 10] Streamlit stop failed:", e)
    else:
        print("[CELL 10] Streamlit was not running")


[CELL 10] STOP_SERVERS_AT_END=False -> keeping FastAPI/Streamlit running.
[CELL 10] Set STOP_SERVERS_AT_END=True in Cell 1 and rerun Cell 10 when you want to stop services.
